# 09 — MetricAdvisor: automatic MonitoringPlan generation

`MetricAdvisor` analyses your data and builds a ready-to-use `MonitoringPlan` — no manual
metric selection needed.

| Topic | What it covers |
|---|---|
| `MetricAdvisor.suggest()` | Basic usage — generate a plan from a DataFrame |
| Class imbalance routing | How imbalance ratio reshapes performance metrics |
| Reference-based routing | Variance ratio → Levene's test |
| Regression task | `task_type='regression'` |
| End-to-end | Use the suggested plan directly with `Runner` |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from ayn_ml.advisor import MetricAdvisor
from ayn_ml.core.schema import TabularSchema

rng = np.random.default_rng(42)
n = 600

# Reference window — stable baseline
df_ref = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_prob": rng.uniform(0.1, 0.9, size=n),
        # numeric — normal distribution
        "age": rng.normal(40, 8, size=n),
        # numeric — right-skewed (log-normal)
        "income": rng.lognormal(10.8, 0.5, size=n),
        # categorical
        "region": rng.choice(["north", "south", "east", "west"], size=n),
        # binary (0/1 encoded)
        "has_prior_claim": rng.integers(0, 2, size=n),
    }
)

# Current window — age slightly shifted, income variance increased
df_cur = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_prob": rng.uniform(0.1, 0.9, size=n),
        "age": rng.normal(43, 9, size=n),  # mean shift +3
        "income": rng.lognormal(10.8, 0.85, size=n),  # variance increase → triggers Levene
        "region": rng.choice(["north", "south", "east", "west"], size=n),
        "has_prior_claim": rng.integers(0, 2, size=n),
    }
)

schema = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_prob",
)
designer = MetricAdvisor(schema)

print(f"Reference : {df_ref.shape}")
print(f"Current   : {df_cur.shape}")

## 1. Basic usage

Pass the current DataFrame and the reference baseline to `suggest()`. The advisor
inspects every feature column — routing normality and skewness tests on the reference
distribution — and builds a `MonitoringPlan` with drift and performance metrics.

In [ ]:
result = designer.suggest(
    df_cur,
    reference=df_ref,
    task_type="classification",
    name="insurance_monitor",
)

print(f"Generated {len(result.plan.metrics)} metric specs")
print(f"{len(result.warnings)} advisory warnings")

In [ ]:
# Advisory warnings — routing decisions explained in plain text
for w in result.warnings:
    print("⚠", w)

In [ ]:
# Inspect the generated specs — grouped by category
from collections import defaultdict

groups = defaultdict(list)
for spec in result.plan.metrics:
    label = spec.feature_name or "(global)"
    groups[label].append(spec.name)

for feature, names in sorted(groups.items()):
    print(f"  {feature:20s}  {', '.join(names)}")

## 2. Metric count by type

Inspect which metric types the advisor selects and how many specs are generated.

In [ ]:
r = designer.suggest(df_cur, reference=df_ref, task_type="classification")
print(f"Total: {len(r.plan.metrics)} metric specs\n")

from collections import Counter
type_counts = Counter(s.metric_type.value for s in r.plan.metrics)
for mtype, count in sorted(type_counts.items()):
    print(f"  {mtype:20s}  {count}")

In [ ]:
# Inspect specs generated for each feature
from collections import defaultdict

groups = defaultdict(list)
for spec in r.plan.metrics:
    label = spec.feature_name or "(global)"
    groups[label].append(spec.name)

for feature, names in sorted(groups.items()):
    print(f"  {feature:20s}  {', '.join(names)}")

## 3. Class imbalance routing

The advisor computes the max/min class ratio on the current window. When imbalance is
detected, `accuracy` is demoted or excluded and `f1` + `aucpr` / `auc` are promoted.

| Ratio | Effect |
|---|---|
| ≤ 5:1 | Balanced — `accuracy` primary |
| > 5:1 | `accuracy` demoted to supplementary, warning emitted |
| > 10:1 | `accuracy` excluded — severe imbalance; `aucpr` promoted |

In [ ]:
# Build three datasets with increasing imbalance
for ratio_desc, n_pos, n_neg in [
    ("balanced  (2:1)", 200, 400),
    ("moderate  (7:1)", 70, 530),
    ("severe   (15:1)", 37, 563),
]:
    labels = np.concatenate([np.ones(n_pos, dtype=int), np.zeros(n_neg, dtype=int)])
    rng.shuffle(labels)
    df_imb = pd.DataFrame(
        {
            "y_true": labels,
            "y_pred": labels,  # perfect — not relevant here
            "y_prob": rng.uniform(0, 1, size=600),
            "age": rng.normal(40, 8, size=600),
        }
    )
    r = designer.suggest(df_imb, reference=df_ref, task_type="classification")
    perf = [s.name for s in r.plan.metrics if s.feature_name is None and s.name not in ("target_drift",)]
    warns = list(r.warnings)
    print(f"{ratio_desc}  →  perf specs: {perf}")
    for w in warns:
        print(f"                ↳ {w}")

## 4. Normality routing and Levene's test

`MetricAdvisor` runs normality and skewness tests on the **reference** distribution — the
stable baseline you are monitoring against. This avoids a circular dependency: using the
potentially-drifted current window for normality routing could silently change the test
choice (e.g. a mean shift could make the current window non-normal).

For each numeric column the advisor also computes `variance_ratio = std_current / std_reference`.
Levene's test is added (and a warning emitted) when the ratio is outside the `[0.67, 1.5]` band.

In [ ]:
# income: std_cur ≈ std_ref × 1.7 — outside [0.67, 1.5] → Levene triggered
# age:    std_cur ≈ std_ref × 1.1 — within band → no Levene
import numpy as np

income_ratio = df_cur["income"].std() / df_ref["income"].std()
age_ratio = df_cur["age"].std() / df_ref["age"].std()


def _levene_label(ratio):
    return "→ Levene" if ratio > 1.5 or ratio < 0.67 else "→ no Levene"


print(f"income variance_ratio: {income_ratio:.2f}  {_levene_label(income_ratio)}")
print(f"age    variance_ratio: {age_ratio:.2f}  {_levene_label(age_ratio)}")
print()

result = designer.suggest(df_cur, reference=df_ref, task_type="classification")

levene_features = [s.feature_name for s in result.plan.metrics if s.name == "levene"]
print(f"Levene added for: {levene_features}")
print()
for w in result.warnings:
    print("⚠", w)

## 5. Small-sample guard (n < 30)

With fewer than 30 non-null rows in the current window, hypothesis tests are unreliable.
The advisor falls back to Wasserstein only and emits a warning. The reference is still
required (for normality routing and variance ratio), but the sample-size routing is based
on the current window.

In [ ]:
df_tiny = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=20),
        "y_pred": rng.integers(0, 2, size=20),
        "y_prob": rng.uniform(0, 1, size=20),
        "age": rng.normal(40, 8, size=20),
    }
)

df_ref_tiny = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=200),
        "y_pred": rng.integers(0, 2, size=200),
        "y_prob": rng.uniform(0, 1, size=200),
        "age": rng.normal(40, 8, size=200),
    }
)
r_tiny = designer.suggest(df_tiny, reference=df_ref_tiny, task_type="classification")

drift_for_age = [s.name for s in r_tiny.plan.metrics if s.feature_name == "age"]
print(f"Drift specs for 'age' (n=20): {drift_for_age}")
for w in r_tiny.warnings:
    print("⚠", w)

## 6. Regression task

In [ ]:
schema_reg = TabularSchema(label_col="y_true", prediction_col="y_pred")
designer_reg = MetricAdvisor(schema_reg)

df_reg = pd.DataFrame(
    {
        "y_true": rng.normal(0, 1, size=400),
        "y_pred": rng.normal(0, 1, size=400),
        "feature_a": rng.normal(5, 2, size=400),
        "category": rng.choice(["A", "B", "C"], size=400),
    }
)
df_ref_reg = pd.DataFrame(
    {
        "y_true": rng.normal(0, 1, size=400),
        "y_pred": rng.normal(0, 1, size=400),
        "feature_a": rng.normal(5, 2, size=400),
        "category": rng.choice(["A", "B", "C"], size=400),
    }
)

r = designer_reg.suggest(df_reg, reference=df_ref_reg, task_type="regression")
perf = [s.name for s in r.plan.metrics if s.feature_name is None and s.name not in ("target_drift",)]
print(f"Regression perf metrics: {perf}")

## 7. End-to-end — suggested plan → Runner

The generated plan is a standard `MonitoringPlan` — pass it directly to `Runner`.

In [ ]:
from ayn_ml.runner import Runner

result = designer.suggest(
    df_cur,
    reference=df_ref,
    task_type="classification",
    name="insurance_monitor",
    model_id="ins_model",
    model_version="3.1",
)

report = Runner(strict=False).run(
    result.plan,
    current=df_cur,
    reference=df_ref,
)

print(f"Metrics computed : {len(report.results)}")
print(f"Errors           : {len(report.errors)}")
report.to_dataframe()[["metric_name", "feature_name", "value", "status"]].head(20)

## 8. SuggestedPlan serialisation

`SuggestedPlan.to_dict()` returns a JSON-compatible dict — useful for logging the advisor's
decisions alongside the monitoring run.

In [ ]:
d = result.to_dict()
print("Top-level keys:", list(d.keys()))
print("Warnings:")
for w in d["warnings"]:
    print(" -", w)

# Round-trip the plan through JSON
from ayn_ml.core.spec import MonitoringPlan

plan_rt = MonitoringPlan.model_validate(d["plan"])
assert len(plan_rt.metrics) == len(result.plan.metrics)
print(f"\nRound-trip OK — {len(plan_rt.metrics)} specs")